# Notebook 04 — Linearization and Taylor Approximation (EKF intuition)

**Goal:** Understand why EKF linearizes nonlinear functions and how first-order Taylor approximations behave.

**Rookie focus:** We'll keep the math gentle, visualize everything, and always connect back to EKF-SLAM.


## Table of Contents

- [1) Why linearization is needed in EKF](#1-why-linearization-is-needed-in-ekf)
- [2) First-order Taylor approximation](#2-first-order-taylor-approximation)
- [3) Approximation error visualization](#3-approximation-error-visualization)
- [4) Interactive examples (edit operating point and rerun)](#4-interactive-examples-edit-operating-point-and-rerun)
- [5) Exercise — compare expansion points](#5-exercise--compare-expansion-points)
- [6) EKF connection (must-remember)](#6-ekf-connection-must-remember)

---

## Navigation

- Previous: [ntbk-03-covariance-ellipses-and-geometry.ipynb](./ntbk-03-covariance-ellipses-and-geometry.ipynb)
- Next: [ntbk-05-jacobians-for-robotics.ipynb](./ntbk-05-jacobians-for-robotics.ipynb)

## Relevant source files (repo paths)
- `src/kiss_slam/models/motion.py`
- `src/kiss_slam/models/measurement.py`


In [ ]:
# Notebook setup (reproducible + matplotlib defaults)
import numpy as np
import matplotlib.pyplot as plt
from _notebook_utils import set_seed

SEED = 103
set_seed(SEED)
plt.rcdefaults()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)


## 1) Why linearization is needed in EKF

In EKF-SLAM we use nonlinear functions:
- **Motion model:** robot kinematics are nonlinear.
- **Measurement model:** range/bearing from robot to landmark is nonlinear.

A standard Kalman filter assumes linear models. EKF handles this by:
1. Choosing an operating point (current estimate),
2. Replacing a nonlinear function with its **first-order Taylor approximation**,
3. Using the local slope (Jacobian) as a linear model.

So linearization is not optional in EKF—it is the bridge from nonlinear robotics to Kalman math.


## 2) First-order Taylor approximation

For a scalar function \(f(x)\), linearization around \(x_0\) is:

\[
f(x) \approx f(x_0) + f'(x_0)(x - x_0)
\]

Interpretation:
- \(f(x_0)\): value at the operating point,
- \(f'(x_0)\): local slope,
- approximation quality is best near \(x_0\).


In [ ]:
def taylor_first_order_scalar(f, df, x, x0):
    return f(x0) + df(x0) * (x - x0)

# Example: sin(x)
f = np.sin
df = np.cos
x = np.linspace(-np.pi, np.pi, 400)
x0 = 0.8

y_true = f(x)
y_lin = taylor_first_order_scalar(f, df, x, x0)

plt.plot(x, y_true, label="sin(x)", linewidth=2)
plt.plot(x, y_lin, "--", label=f"linearized at x0={x0:.2f}")
plt.scatter([x0], [f(x0)], color="k", zorder=5)
plt.title("First-order Taylor approximation")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 3) Approximation error visualization

Define error: \(e(x)=f(x)-\hat{f}(x)\).
In EKF, this local approximation error is one reason filter quality can degrade when:
- uncertainty is large,
- dynamics are strongly nonlinear,
- linearization point is poor.


In [ ]:
def plot_function_and_error(name, f, df, x0, x_min=-3.5, x_max=3.5):
    x = np.linspace(x_min, x_max, 600)
    y = f(x)
    yhat = taylor_first_order_scalar(f, df, x, x0)
    err = y - yhat

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(x, y, label=f"{name}(x)", linewidth=2)
    axes[0].plot(x, yhat, "--", label=f"linearized @ x0={x0:.2f}")
    axes[0].scatter([x0], [f(x0)], color="k", s=40)
    axes[0].set_title(f"{name}(x): true vs linearized")
    axes[0].set_xlabel("x")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(x, err, color="tab:red", label="error = true - approx")
    axes[1].axhline(0.0, color="k", linewidth=1)
    axes[1].set_title("Approximation error")
    axes[1].set_xlabel("x")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_function_and_error("sin", np.sin, np.cos, x0=1.2)


## 4) Interactive examples (edit operating point and rerun)

In a notebook, a lightweight way to be interactive is:
- edit `x0` below,
- rerun the cell,
- compare behavior.

We'll test three EKF-relevant nonlinear functions:
1. \(\sin(x)\),
2. \(\arctan2(y, x)\) with fixed \(y\),
3. range \(r(x)=\sqrt{x^2 + y^2}\) with fixed \(y\).


In [ ]:
# --- Change these and rerun this cell ---
x0 = 0.5
y_fixed = 1.0
x_min, x_max = -4.0, 4.0

plot_function_and_error("sin", np.sin, np.cos, x0=x0, x_min=x_min, x_max=x_max)

def f_atan2(x):
    return np.arctan2(y_fixed, x)

def df_atan2(x):
    return -y_fixed / (x**2 + y_fixed**2)

plot_function_and_error("atan2(y_fixed, x)", f_atan2, df_atan2, x0=x0, x_min=x_min, x_max=x_max)

def f_range(x):
    return np.sqrt(x**2 + y_fixed**2)

def df_range(x):
    return x / np.sqrt(x**2 + y_fixed**2)

plot_function_and_error("range(x)", f_range, df_range, x0=x0, x_min=x_min, x_max=x_max)


## 5) Exercise — compare expansion points

**Task:** For each function, compare errors for three operating points (example: `x0 = -2, 0, 2`).

- Which operating point gives the smallest error near the region you care about?
- How does this map to EKF behavior when the state estimate drifts?

<details>
<summary>Optional solution idea</summary>

Use an error metric over a local window, e.g. mean absolute error on \([x_0-0.5, x_0+0.5]\):

```python
def local_mae(f, df, x0, width=0.5, n=200):
    x = np.linspace(x0 - width, x0 + width, n)
    y = f(x)
    yhat = taylor_first_order_scalar(f, df, x, x0)
    return np.mean(np.abs(y - yhat))
```

Then print this for each `x0` and function.

In EKF terms: better operating point -> better linear approximation -> more consistent update.
</details>


## 6) EKF connection (must-remember)

- EKF linearizes nonlinear models around the **current estimate**.
- Jacobians are just the multi-dimensional version of slope in Taylor expansion.
- If your estimate is far from truth, linearization can be poor, harming filter consistency.

Next notebook: how to compute Jacobians in practice (numeric + analytic) and validate them.


## Exercise Solutions

<details>
<summary>Show solution sketches</summary>

- Re-run the exercise code cells and compare your intermediate values to the reference outputs printed in this notebook.
- Check shapes (`mean`, `covariance`, Jacobians) first; most EKF mistakes are shape/sign issues.
- Use the same `SEED` from the setup cell to keep your run deterministic while debugging.

</details>
